# Practical Activity 1: Classical Cryptography and Perfect Encryption

## Objectives
- Implement several **classical ciphers** using Python.  
- Analyse their **security properties**.  
- Understand the concept of **perfect secrecy**.  
- Solve related **problems and questions**.  

## 1. Introduction

Classical cryptography refers to the historical methods of encrypting messages before the rise of modern algorithms.  
Examples include the **Caesar cipher**, the **Affine cipher**, and the **Vigenère cipher**.  

While these systems were groundbreaking for their time, they are **cryptographically weak** by modern standards.  

The **One-Time Pad (OTP)**, however, is a special cipher that provides **perfect secrecy** (in Shannon’s sense) but has practical limitations due to key distribution.  


## 2. Classical Ciphers Implementation

For our code, we will need to use a little bit of modular arithmetic. Concretely, we will need to compute the inverse of a number in $\mathbb{Z}_n$, which is done as follows:

In [94]:
def modinv(a, m):
    """
    Compute the modular inverse of a modulo m.
    Returns x such that (a * x) % m == 1, or None if no inverse exists.
    """
    # Extended Euclidean Algorithm
    def egcd(x, y):
        if y == 0:
            return x, 1, 0
        g, u, v = egcd(y, x % y)
        return g, v, u - (x // y) * v

    g, x, _ = egcd(a, m)
    if g != 1:
        return None  # inverse does not exist
    else:
        return x % m


<strong> Explanation: </strong> 
* Given $a,b$ integers, the <strong>Extended Euclidean Algorithm (EEA)</strong> computes n,m and g such that $an+mb=g$ where $g$ is the greatest common divisor of $a,b$.   
* Given $a,m$ integers, $a=1\mod m$ means that $a=mx+1$. Hence $ab=1\mod m$ iff $ab=my+1$ iff the greatest common divisor of $a,m$ is $1$, and then the EEA computes $b$ (the inverse of $a$ modulo $m$). 
* We need to compute modular inverses for decryption if the affine cypher.

In [95]:
# 2.1 Caesar Cipher
def caesar_encrypt(text, key):
    result = ""
    for char in text.upper():
        if char.isalpha():
            result += chr(((ord(char) - 65 + key) % 26) + 65)
        else:
            result += char
    return result

def caesar_decrypt(cipher, key):
    return caesar_encrypt(cipher, -key)

# Example
plaintext = "Cryptography IS FUN and I Love it!"
cipher = caesar_encrypt(plaintext, 13)
decrypted = caesar_decrypt(cipher, 13)
plaintext, cipher, decrypted

('Cryptography IS FUN and I Love it!',
 'PELCGBTENCUL VF SHA NAQ V YBIR VG!',
 'CRYPTOGRAPHY IS FUN AND I LOVE IT!')

In [96]:
# 2.2 Affine Cipher
def affine_encrypt(text, a, b):
    result = ""
    for char in text.upper():
        if char.isalpha():
            result += chr(((a * (ord(char) - 65) + b) % 26) + 65)
        else:
            result += char
    return result

def affine_decrypt(cipher, a, b):
    result = ""
    inv_a = modinv(a, 26)
    for char in cipher:
        if char.isalpha():
            result += chr(((inv_a * ((ord(char) - 65) - b)) % 26) + 65)
        else:
            result += char
    return result

# Example
cipher = affine_encrypt("HELLO WORLD", 5, 8)
decrypted = affine_decrypt(cipher, 5, 8)
cipher, decrypted

('RCLLA OAPLX', 'HELLO WORLD')

In [97]:
# 2.3 Vigenère Cipher
def vigenere_encrypt(text, key):
    result = ""
    key = key.upper()
    key_index = 0
    for char in text.upper():
        if char.isalpha():
            shift = ord(key[key_index % len(key)]) - 65
            result += chr(((ord(char) - 65 + shift) % 26) + 65)
            key_index += 1
        else:
            result += char
    return result

def vigenere_decrypt(cipher, key):
    result = ""
    key = key.upper()
    key_index = 0
    for char in cipher.upper():
        if char.isalpha():
            shift = ord(key[key_index % len(key)]) - 65
            result += chr(((ord(char) - 65 - shift) % 26) + 65)
            key_index += 1
        else:
            result += char
    return result

# Example

plaintext = (
    "One of the most singular characteristics of the art of deciphering is the strong conviction "
    "possessed by every person, even moderately acquainted with it, that he is able to construct "
    "a cipher which nobody else can decipher. I have also observed that the cleverer the person, "
    "the more intimate is his conviction. ―Charles Babbage, Passages from the Life of a Philosopher"
)

cipher = vigenere_encrypt(plaintext, "RIGHT")
decrypted = vigenere_decrypt(cipher, "RIGHT")

print("Ciphertext:\n", cipher)
print("Decrypted:\n", decrypted)

Ciphertext:
 FVK VY KPK THJB YPGXCRHK TPGYTTBKYBJBOJL FN ZOX RZZ VY UMIPIYMXPGX QY AAV AZYHEO IVGMQIABFV VVLJMYZXU JE LOVZE WXIAUU, XMMT THUMXHMVTE HVHCGPGKMJ DBKP OA, MYIZ OX ZA GIEV BU JHEAZYNTB G JBGPKY PYQIO GFJUKR VTYL VRV JLVZXNLK. Z PGCX RTYV HSAKYOVL ZOTK BNL VCMBLKVZ ZOX GMXZHE, BNL FFZK PGKQSHMV QY OBJ KUUOZKZPHE. ―KNHKCMY ITSJGNX, GIYZTXMY MKFU ZOX CQLL HW I VOBCWYVIYMX
Decrypted:
 ONE OF THE MOST SINGULAR CHARACTERISTICS OF THE ART OF DECIPHERING IS THE STRONG CONVICTION POSSESSED BY EVERY PERSON, EVEN MODERATELY ACQUAINTED WITH IT, THAT HE IS ABLE TO CONSTRUCT A CIPHER WHICH NOBODY ELSE CAN DECIPHER. I HAVE ALSO OBSERVED THAT THE CLEVERER THE PERSON, THE MORE INTIMATE IS HIS CONVICTION. ―CHARLES BABBAGE, PASSAGES FROM THE LIFE OF A PHILOSOPHER


In [98]:
# 2.4 One-Time Pad (OTP) with XOR
import os

# Imports Python’s os module, which includes os.urandom() to generate cryptographically secure random bytes.
# The os module in Python is part of the standard library. 
# It provides functions for interacting with the operating system (OS).
# Think of it as a bridge between Python code and the system resources 
# (like files, directories, environment variables, random number generation, etc.)


def otp_encrypt(text, key):
    return bytes([ord(c) ^ k for c, k in zip(text, key)])


# zip(text, key) pairs each character of the plaintext with one byte of the key.

# ord(c) converts each character c into its ASCII code.

# ord(c) ^ k applies XOR between the plaintext byte and key byte.

# The result is collected into a list and wrapped in bytes([...]) so the ciphertext is stored as raw bytes.



def otp_decrypt(cipher, key):
    return ''.join([chr(c ^ k) for c, k in zip(cipher, key)])

# Note that encryption and decryption are the same in this case. 

# Example
plaintext = (
    "A little bit of math can accomplish what all the guns and barbed wire can't: "
    "a little bit of math can keep a secret."
    "Edward Snowden."
)
key = os.urandom(len(plaintext))  # truly random key
# Indeed, os.urandom(n) generates n cryptographically secure random bytes.
cipher = otp_encrypt(plaintext, key)
decrypted = otp_decrypt(cipher, key)
plaintext, cipher, decrypted

("A little bit of math can accomplish what all the guns and barbed wire can't: a little bit of math can keep a secret.Edward Snowden.",
 b'\xc9F\xdc\xbd\xe3Z\xac\x18x\xde/G\xb0\xd5\x04f\x9f\xd7\x9d\xe7\xf8\xc1`\xed\xff[@5j[\\\xb4\xe1\xfa\x97\x93\xf9e\x95V\x97\x9c\x0c\x96\x11\x81R\x0c\xa8r\x01\xcb\xf5\x03\xcbH;\xc2\xef\x1c\xc0\xe2\xe8\x02\xe8\xd5\x86\x8e/\xc91),\x80^T\xd1\xc3\xce\xae\xa9\xa0\xf6\x01w\x1d3\x8c6\xeffN\x087+\t\xb1\x13\xe4\xbf\xe8\x08$\x1b\xcd\x1e\xb2q{X}\xa2\x0c=\x1e\\d2@\xc9\xa3\xee.\xc7>$ws\x9fQ\x82',
 "A little bit of math can accomplish what all the guns and barbed wire can't: a little bit of math can keep a secret.Edward Snowden.")

Let us now put everything in bits, since OTP and XOR run with bits...:

In [99]:
def to_bits(data):
    """Convert a bytes or string object into a string of bits."""
    if isinstance(data, str):
        data = data.encode("utf-8")  # convert string to bytes
    return ''.join(f"{byte:08b}" for byte in data)

def otp_encrypt(text, key):
    """Encrypt plaintext (string) with key (bytes)."""
    return bytes([ord(c) ^ k for c, k in zip(text, key)])

def otp_decrypt(cipher, key):
    """Decrypt ciphertext (bytes) with key (bytes)."""
    return ''.join([chr(c ^ k) for c, k in zip(cipher, key)])


# Example
plaintext = (
    "A little bit of math can accomplish what all the guns and barbed wire can't: "
    "a little bit of math can keep a secret."
    "Edward Snowden."
)
key = os.urandom(len(plaintext))  # random key (bytes)
cipher = otp_encrypt(plaintext, key)
decrypted = otp_decrypt(cipher, key)

print("Plaintext (ASCII):", plaintext)
print("Plaintext (bits): ", ' '.join(f"{ord(c):08b}" for c in plaintext))

print("\nKey (bits):       ", ' '.join(f"{k:08b}" for k in key))

print("\nCipher (bits):    ", ' '.join(f"{c:08b}" for c in cipher))
print("Cipher (raw):     ", cipher)

print("\nDecrypted (bits): ", ' '.join(f"{ord(c):08b}" for c in decrypted))
print("Decrypted (text):", decrypted)


Plaintext (ASCII): A little bit of math can accomplish what all the guns and barbed wire can't: a little bit of math can keep a secret.Edward Snowden.
Plaintext (bits):  01000001 00100000 01101100 01101001 01110100 01110100 01101100 01100101 00100000 01100010 01101001 01110100 00100000 01101111 01100110 00100000 01101101 01100001 01110100 01101000 00100000 01100011 01100001 01101110 00100000 01100001 01100011 01100011 01101111 01101101 01110000 01101100 01101001 01110011 01101000 00100000 01110111 01101000 01100001 01110100 00100000 01100001 01101100 01101100 00100000 01110100 01101000 01100101 00100000 01100111 01110101 01101110 01110011 00100000 01100001 01101110 01100100 00100000 01100010 01100001 01110010 01100010 01100101 01100100 00100000 01110111 01101001 01110010 01100101 00100000 01100011 01100001 01101110 00100111 01110100 00111010 00100000 01100001 00100000 01101100 01101001 01110100 01110100 01101100 01100101 00100000 01100010 01101001 01110100 00100000 01101111 01100110 00

## 3. Cryptanalysis Exercises

In [100]:
# 3.1 Brute force attack on Caesar cipher
cipher = caesar_encrypt("MEET ME AT MIDNIGHT", 7)
print("Cipher:", cipher)

print("\nBrute force results:")
for k in range(26):
    print(f"Key {k}: {caesar_decrypt(cipher, k)}")

Cipher: TLLA TL HA TPKUPNOA

Brute force results:
Key 0: TLLA TL HA TPKUPNOA
Key 1: SKKZ SK GZ SOJTOMNZ
Key 2: RJJY RJ FY RNISNLMY
Key 3: QIIX QI EX QMHRMKLX
Key 4: PHHW PH DW PLGQLJKW
Key 5: OGGV OG CV OKFPKIJV
Key 6: NFFU NF BU NJEOJHIU
Key 7: MEET ME AT MIDNIGHT
Key 8: LDDS LD ZS LHCMHFGS
Key 9: KCCR KC YR KGBLGEFR
Key 10: JBBQ JB XQ JFAKFDEQ
Key 11: IAAP IA WP IEZJECDP
Key 12: HZZO HZ VO HDYIDBCO
Key 13: GYYN GY UN GCXHCABN
Key 14: FXXM FX TM FBWGBZAM
Key 15: EWWL EW SL EAVFAYZL
Key 16: DVVK DV RK DZUEZXYK
Key 17: CUUJ CU QJ CYTDYWXJ
Key 18: BTTI BT PI BXSCXVWI
Key 19: ASSH AS OH AWRBWUVH
Key 20: ZRRG ZR NG ZVQAVTUG
Key 21: YQQF YQ MF YUPZUSTF
Key 22: XPPE XP LE XTOYTRSE
Key 23: WOOD WO KD WSNXSQRD
Key 24: VNNC VN JC VRMWRPQC
Key 25: UMMB UM IB UQLVQOPB


In [101]:
# 3.2 Frequency analysis helper
from collections import Counter

def frequency_analysis(text):
    text = ''.join([c for c in text.upper() if c.isalpha()])
    counts = Counter(text)
    total = sum(counts.values())
    return {c: round(counts[c] / total, 3) for c in counts}

cipher = caesar_encrypt("THIS IS A SECRET MESSAGE THAT WE WILL TRY TO BREAK", 5)
frequency_analysis(cipher)

{'Y': 0.15,
 'M': 0.05,
 'N': 0.075,
 'X': 0.125,
 'F': 0.1,
 'J': 0.15,
 'H': 0.025,
 'W': 0.075,
 'R': 0.025,
 'L': 0.025,
 'B': 0.05,
 'Q': 0.05,
 'D': 0.025,
 'T': 0.025,
 'G': 0.025,
 'P': 0.025}

<strong> Index of Coincidence (IC) in Vigenére Cryptanalysis</strong>

The Index of Coincidence (IC) is a statistical measure that estimates how likely it is that two randomly chosen letters from a text are the same.

Formally, for a text of length $N$, with letter counts 
$f_A, f_B, \dots, f_Z$, the IC is defined as:
$$ IC = \frac{\sum_{i=A}^{Z} f_i (f_i - 1)}{N(N-1)}.$$

Here:
* $f_i$ = frequency of letter $i$,
* $N$ = total number of letters in the text.


Intuition: 
* If the text is completely random, all letters appear with probability $\tfrac{1}{26}$.
    $$\text{Expected } IC \approx \frac{1}{26} \approx 0.038.$$
* If the text is English plaintext, letters follow natural frequencies (E, T, A are more common).
   $$\text{Expected } IC \approx 0.066.$$


So:
$$\text{Random text} \;\;\Rightarrow\;\; IC \approx 0.038$$

$$\text{English text} \;\;\Rightarrow\;\; IC \approx 0.066$$

* Application to the Vigenére Cipher: 

The Vigenére cipher shifts the plaintext with a periodic key of length $k$.

* If we guess a key length $k$ and split the ciphertext into $k$ subsequences (taking every $k$-th letter), then:
    
*  If $k$ is correct, each subsequence is essentially a Caesar cipher of English text, so its IC is closer to $0.066$.
*  If $k$ is wrong, subsequences mix multiple Caesar shifts, so they look more random and the IC is closer to $0.038$.

Workflow:
*  For each candidate key length $k$:

        * Split ciphertext into subsequences.
        * Compute IC for each subsequence.
        * Take the average IC.
Compare results:
* Peaks closer to $0.066$ suggest the likely key length.
* Flat values near $0.038$ suggest incorrect lengths.
   
   
   Key Insight: 
The Index of Coincidence works because it distinguishes between:
* Natural language distribution: biased toward some letters, higher IC.
* Uniform random distribution: all letters equal, lower IC.


Thus, IC is a statistical tool to guess the Vigenére key length.


In [102]:
# 3.3 Index of Coincidence (IC) for Vigenère analysis
def index_of_coincidence(text):
    text = ''.join([c for c in text.upper() if c.isalpha()])
    N = len(text)
    freqs = Counter(text)
    ic = sum([f*(f-1) for f in freqs.values()]) / (N*(N-1))
    return ic

sample = vigenere_encrypt("THIS IS A LONGER TEXT TO COMPUTE INDEX OF COINCIDENCE", "KEY")
index_of_coincidence(sample)

0.05813953488372093

Note that spaces do not affect to the computation above!:

In [103]:
sample = vigenere_encrypt("THISISALONGERTEXTTOCOMPUTEINDEXOFCOINCIDENCE", "KEY")
index_of_coincidence(sample)

0.05813953488372093

## 4. Questions & Problems (Solved)

1. **Show that Caesar cipher can be brute-forced easily**  
   → Done in section 3.1, only 26 possibilities.  

2. **Why is the Affine cipher vulnerable to frequency analysis?**  
   → Because it’s a substitution cipher with a fixed mapping, so letter frequencies remain.  

3. **Implement a function that guesses Vigenère key length using IC.**  

4. **Challenge: Encrypt with Vigenère and break it using frequency analysis.**  
   → Implemented with IC-based key length guess, then frequency analysis

In [112]:
from collections import Counter

def clean_text(text):
    """Keep only uppercase A-Z characters."""
    return ''.join(filter(str.isalpha, text.upper()))

def index_of_coincidence(text):
    """Calculates IC for a string of length N."""
    N = len(text)
    if N <= 1:
        return 0.0
    counts = Counter(text)
    return sum(f * (f - 1) for f in counts.values()) / (N * (N - 1))

def autocorrelation_score(cipher, shift):
    """Counts matching characters when ciphertext is shifted by 'shift' places."""
    clean_c = clean_text(cipher)
    matches = sum(1 for i in range(len(clean_c) - shift) if clean_c[i] == clean_c[i + shift])
    return matches / (len(clean_c) - shift)

def guess_vigenere_key_length(cipher, max_len=10):
    clean_c = clean_text(cipher)
    
    ic_scores = {}
    auto_scores = {}
    combined_scores = {}
    
    for key_len in range(1, max_len + 1):
        # 1. Calculate Average IC for this key length
        ic_values = []
        for i in range(key_len):
            subseq = clean_c[i::key_len]
            ic_values.append(index_of_coincidence(subseq))
        
        avg_ic = sum(ic_values) / len(ic_values) if ic_values else 0
        ic_scores[key_len] = avg_ic
        
        # 2. Calculate Autocorrelation Score
        auto_score = autocorrelation_score(clean_c, key_len)
        auto_scores[key_len] = auto_score
        
        # 3. Combined metric (IC * Autocorrelation)
        combined_scores[key_len] = avg_ic * auto_score

    # Determine best guess
    best_key_len = max(combined_scores, key=combined_scores.get)
    
    return {
        "best_guess": best_key_len,
        "combined_scores": combined_scores,
        "ic_scores": ic_scores,
        "autocorrelation_scores": auto_scores
    }

In [113]:
plaintext=(
    "Anyone who tries to create his or her own cryptographic primitive is either a genius or a fool."
    "Given the genius/fool ratio of our species, the odds aren not very good."
    "Bruce Schneier, Secrets and Lies: Digital Security in a Networked World."
)

cipher = vigenere_encrypt(plaintext, "SAL")
results = guess_vigenere_key_length(cipher, max_len=5)

print(f"Guessed Key Length: {results['best_guess']}")
print("\nIC Scores:", results['ic_scores'])

Guessed Key Length: 3

IC Scores: {1: 0.04436528497409326, 2: 0.044168249231325736, 3: 0.06841931216931217, 4: 0.04107414242292661, 5: 0.045810993179414235}


Now the lenght was guessed : k=3. 

Once the key length $k$ is known, the standard approach to crack a Vigenère cipher is:

* Split the ciphertext into $k$ groups, where each group corresponds to letters encrypted with the same Caesar shift.

* For each group, guess the Caesar shift by comparing letter frequencies with English letter frequencies.

* Collect all shifts → reconstruct the key.

* Decrypt the ciphertext.

Sometimes, the method above can't guess the right k. Then we can just try with several choices of k (that is assumed small, so only a few checks are necessary)

In [114]:
import string
from collections import Counter

# English letter frequencies (approximate, normalized)
ENGLISH_FREQ = {
    'A': 0.082, 'B': 0.015, 'C': 0.028, 'D': 0.043, 'E': 0.13,
    'F': 0.022, 'G': 0.020, 'H': 0.061, 'I': 0.070, 'J': 0.0015,
    'K': 0.0077, 'L': 0.040, 'M': 0.024, 'N': 0.067, 'O': 0.075,
    'P': 0.019, 'Q': 0.00095, 'R': 0.060, 'S': 0.063, 'T': 0.091,
    'U': 0.028, 'V': 0.0098, 'W': 0.024, 'X': 0.0015, 'Y': 0.020, 'Z': 0.00074
}

LETTERS = string.ascii_uppercase

def sanitize(text):
    """Keep only A–Z, and uppercase everything."""
    return ''.join(c for c in text.upper() if c in LETTERS)

def vigenere_encrypt(plaintext, key):
    """Encrypt Vigenère cipher (ignores non-letters)."""
    plaintext = sanitize(plaintext)
    key = key.upper()
    ciphertext = []
    for i, c in enumerate(plaintext):
        shift = LETTERS.index(key[i % len(key)])
        p = (LETTERS.index(c) + shift) % 26
        ciphertext.append(LETTERS[p])
    return ''.join(ciphertext)

def vigenere_decrypt_clean(ciphertext, key):
    """Decrypt ciphertext to clean A–Z only (for analysis)."""
    ciphertext = sanitize(ciphertext)
    plaintext = []
    key = key.upper()
    for i, c in enumerate(ciphertext):
        shift = LETTERS.index(key[i % len(key)])
        p = (LETTERS.index(c) - shift) % 26
        plaintext.append(LETTERS[p])
    return ''.join(plaintext)

def vigenere_encrypt_with_punct(plaintext, key):
    """
    Encrypt Vigenère cipher while preserving non-letters.
    Only A–Z are shifted; everything else is copied.
    """
    ciphertext = []
    key = key.upper()
    j = 0  # index for key
    for c in plaintext.upper():
        if c in LETTERS:
            shift = LETTERS.index(key[j % len(key)])
            p = (LETTERS.index(c) + shift) % 26
            ciphertext.append(LETTERS[p])
            j += 1
        else:
            ciphertext.append(c)  # keep punctuation, digits, spaces
    return ''.join(ciphertext)

def vigenere_decrypt_with_punct(ciphertext, key):
    """
    Decrypt Vigenère cipher while preserving non-letters.
    Only A–Z are shifted back; everything else is copied.
    """
    plaintext = []
    key = key.upper()
    j = 0  # index for key
    for c in ciphertext.upper():
        if c in LETTERS:
            shift = LETTERS.index(key[j % len(key)])
            p = (LETTERS.index(c) - shift) % 26
            plaintext.append(LETTERS[p])
            j += 1
        else:
            plaintext.append(c)
    return ''.join(plaintext)

def caesar_score(text):
    """Score a text segment by comparing letter frequencies with English frequencies."""
    N = len(text)
    if N == 0:
        return float('inf')
    counts = Counter(text)
    chi2 = 0
    for letter in LETTERS:
        observed = counts.get(letter, 0)
        expected = ENGLISH_FREQ[letter] * N
        chi2 += (observed - expected) ** 2 / (expected + 1e-9)
    return chi2

def guess_caesar_shift(segment):
    """Guess the Caesar shift for a segment using chi-squared statistic."""
    segment = sanitize(segment)
    best_shift, best_score = None, float('inf')
    for shift in range(26):
        decrypted = ''.join(LETTERS[(LETTERS.index(c) - shift) % 26] for c in segment)
        score = caesar_score(decrypted)
        if score < best_score:
            best_score = score
            best_shift = shift
    return best_shift

def guess_vigenere_key(ciphertext, key_len):
    """Guess the Vigenère key of given length."""
    ciphertext = sanitize(ciphertext)
    key = ""
    for i in range(key_len):
        segment = ciphertext[i::key_len]  # every i-th letter
        shift = guess_caesar_shift(segment)
        key += LETTERS[shift]
    return key

# =====================
# Example
# =====================

plaintext=(
    "Anyone who tries to create his or her own cryptographic primitive is either a genius or a fool."
    "Given the genius/fool ratio of our species, the odds aren not very good."
    "Bruce Schneier, Secrets and Lies: Digital Security in a Networked World."
)

ciphertext = vigenere_encrypt_with_punct(plaintext, "sal")

key_len = 3  # assume we already guessed correctly
guessed_key = guess_vigenere_key(ciphertext, key_len)

decrypted_clean = vigenere_decrypt_clean(ciphertext, guessed_key)
decrypted_with_punct = vigenere_decrypt_with_punct(ciphertext, guessed_key)

print("Ciphertext (first 100):", ciphertext)
print("Guessed Key:", guessed_key)
print("\nDecrypted (clean):", decrypted_clean)
print("\nDecrypted (with punctuation):", decrypted_with_punct)


Ciphertext (first 100): SNJGNP OHZ LRTWS EG CCWAEW HTK OC ZEC GWY URJHTZYRLHHTU PCAMTLIGW ID WIEZEC S GPFIFK OC S FZGL.RAVPF TSW GPFIFK/FZGL CSTTG OQ GUC KPPUIPK, TSW OOVS LJEY FOE NECQ GZGD.MJUNW SNZNPAEC, KENJEEK AYV LTWS: OAGTLAW KENMRTLY TF A YWTHGRVWD HGRWV.
Guessed Key: SAL

Decrypted (clean): ANYONEWHOTRIESTOCREATEHISORHEROWNCRYPTOGRAPHICPRIMITIVEISEITHERAGENIUSORAFOOLGIVENTHEGENIUSFOOLRATIOOFOURSPECIESTHEODDSARENNOTVERYGOODBRUCESCHNEIERSECRETSANDLIESDIGITALSECURITYINANETWORKEDWORLD

Decrypted (with punctuation): ANYONE WHO TRIES TO CREATE HIS OR HER OWN CRYPTOGRAPHIC PRIMITIVE IS EITHER A GENIUS OR A FOOL.GIVEN THE GENIUS/FOOL RATIO OF OUR SPECIES, THE ODDS AREN NOT VERY GOOD.BRUCE SCHNEIER, SECRETS AND LIES: DIGITAL SECURITY IN A NETWORKED WORLD.


In [115]:
# Example 2: Try the same with the following plaintext:
plaintext=(
    "It was the best of times, it was the worst of times, "
    "it was the age of wisdom, it was the age of foolishness, "
    "it was the epoch of belief, it was the epoch of incredulity, "
    "it was the season of Light, it was the season of Darkness, "
    "it was the spring of hope, it was the winter of despair."
)
# and with a key of 6 letters. 

# Example 3: Do the same with:

plaintext=(
    "It is insufficient to protect ourselves with laws;" 
    "we need to protect ourselves with mathematics."
)
# and a key of 5 letters.

## 5. Conclusions

- **Classical ciphers** are useful for learning but offer **no real security** today.  
- **Brute force** and **frequency analysis** break them quickly.  
- **Perfect secrecy** exists only with OTP, but key distribution makes it impractical.  
- This motivates **modern cryptography**, which balances strong security with efficiency.  
